# 04 — Model Training V2
## EfficientNetV2-B0 · 4-Class Eye Disease Classification

**Classes:** 0=Normal · 1=Cataract · 2=Diabetic Retinopathy · 3=Glaucoma

Trains a brand-new 4-class model using ImageNet-pretrained EfficientNetV2-B0 with two-phase transfer learning. Uses outputs from `03_dataset_preparation_v2.ipynb`.

**Does NOT load or continue the previous 5-class model.**

In [2]:
# Cell 1 — Imports and configuration
import os, json, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ── Reproducibility ──
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Paths ──
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == 'notebooks' else NOTEBOOK_DIR

PREPROC_DIR  = PROJECT_DIR / 'preprocessing' / 'splits_v2'
MODEL_DIR    = PROJECT_DIR / 'model'
REPORT_DIR   = PROJECT_DIR / 'reports' / 'model_v2'
LOG_DIR      = PROJECT_DIR / 'logs' / 'model_v2'

for d in [MODEL_DIR, REPORT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = PREPROC_DIR / 'train.csv'
VAL_CSV   = PREPROC_DIR / 'validation.csv'
TEST_CSV  = PREPROC_DIR / 'test.csv'

# ── Class mapping (single source of truth) ──
CLASS_NAMES = {0: 'Normal', 1: 'Cataract', 2: 'Diabetic Retinopathy', 3: 'Glaucoma'}
NUM_CLASSES = len(CLASS_NAMES)

# ── Model / training config ──
IMG_SIZE    = 224
BATCH_SIZE  = 32
PHASE1_LR   = 1e-3
PHASE2_LR   = 1e-5
PHASE1_EPOCHS = 20
PHASE2_EPOCHS = 40
DROPOUT_RATE  = 0.35

BEST_MODEL_PATH  = MODEL_DIR / 'efficientnetv2_b0_4class_best.keras'
FINAL_MODEL_PATH = MODEL_DIR / 'efficientnetv2_b0_4class_final.keras'
HISTORY_CSV      = MODEL_DIR / 'training_history_v2.csv'
METADATA_PATH    = MODEL_DIR / 'model_metadata_v2.json'

print('Project dir  :', PROJECT_DIR)
print('TF version   :', tf.__version__)
print('GPU devices  :', tf.config.list_physical_devices('GPU'))
print('Class map    :', CLASS_NAMES)
print('Batch size   :', BATCH_SIZE)
print('Phase1 epochs:', PHASE1_EPOCHS, '| Phase2 epochs:', PHASE2_EPOCHS)


Project dir  : d:\Practice Projects\Disease Detection
TF version   : 2.21.0
GPU devices  : []
Class map    : {0: 'Normal', 1: 'Cataract', 2: 'Diabetic Retinopathy', 3: 'Glaucoma'}
Batch size   : 32
Phase1 epochs: 20 | Phase2 epochs: 40


## Stage 1 — Load split CSVs and analyse class distribution

In [3]:
# Cell 2 — Load split CSVs and compute class weights
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

dist = train_df.groupby(['class_id','disease_name']).size().reset_index(name='count')
dist['pct'] = (dist['count'] / dist['count'].sum() * 100).round(1)
print('\nTraining class distribution:')
print(dist.to_string(index=False))

# Compute class weights to handle imbalance
y_train = train_df['class_id'].values
cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
CLASS_WEIGHTS = {i: float(round(w, 4)) for i, w in enumerate(cw)}
print('\nClass weights:', CLASS_WEIGHTS)

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([CLASS_NAMES[i] for i in sorted(CLASS_NAMES)],
       [dist[dist.class_id==i]['count'].values[0] for i in sorted(CLASS_NAMES)],
       color=['steelblue','darkorange','green','crimson'], alpha=0.85)
ax.set_ylabel('Image count')
ax.set_title('Training set class distribution')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'train_class_distribution.png', dpi=120)
plt.show()
print('Saved train_class_distribution.png')


Train: 4119 | Val: 883 | Test: 883

Training class distribution:
 class_id         disease_name  count  pct
        0               Normal   1596 38.7
        1             Cataract    727 17.6
        2 Diabetic Retinopathy    768 18.6
        3             Glaucoma   1028 25.0

Class weights: {0: 0.6452, 1: 1.4164, 2: 1.3408, 3: 1.0017}
Saved train_class_distribution.png


## Stage 2 — Build tf.data pipelines

In [15]:
# Cell 3 — tf.data pipeline with augmentation on train only
AUTOTUNE = tf.data.AUTOTUNE

augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name='augmentation')

def load_image(path, label):
    raw  = tf.io.read_file(path)
    img  = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img  = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img  = tf.cast(img, tf.float32)   # EfficientNetV2 include_preprocessing=True handles scaling
    return img, label

def make_dataset(df, training=False):
    paths  = df['image_path'].values
    labels = df['class_id'].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(df), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds   = make_dataset(val_df,   training=False)
test_ds  = make_dataset(test_df,  training=False)

print('Train batches :', len(train_ds))
print('Val batches   :', len(val_ds))
print('Test batches  :', len(test_ds))

# Sanity-check one batch
for imgs, lbls in train_ds.take(1):
    print(f'Batch shape: {imgs.shape}, dtype: {imgs.dtype}, '
          f'label range: {lbls.numpy().min()}–{lbls.numpy().max()}')


Train batches : 129
Val batches   : 28
Test batches  : 28
Batch shape: (32, 224, 224, 3), dtype: <dtype: 'float32'>, label range: 0–3


## Stage 3 — Build EfficientNetV2-B0 model

In [5]:
# Cell 4 — Build 4-class EfficientNetV2-B0 model
def build_model(num_classes=NUM_CLASSES, img_size=IMG_SIZE,
                 dropout_rate=DROPOUT_RATE, trainable_backbone=False):
    inputs  = keras.Input(shape=(img_size, img_size, 3), name='image_input')
    backbone = keras.applications.EfficientNetV2B0(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs,
        include_preprocessing=True,   # handles [0,255] -> model-expected range
    )
    backbone.trainable = trainable_backbone

    x = backbone.output
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='head_bn')(x)
    x = layers.Dropout(dropout_rate, name='head_dropout')(x)
    outputs = layers.Dense(num_classes, activation='softmax',
                           name='disease_output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs,
                        name='efficientnetv2b0_4class')
    return model, backbone

model, backbone = build_model(trainable_backbone=False)

# Verify backbone layer name for future Grad-CAM use
backbone_name = backbone.name
print('Backbone layer name :', backbone_name)
print('Total params        :', model.count_params())
print('Trainable params    :', sum(tf.size(v).numpy() for v in model.trainable_variables))
print('Non-trainable params:', sum(tf.size(v).numpy() for v in model.non_trainable_variables))
model.summary(line_length=90, show_trainable=True)


Backbone layer name : efficientnetv2-b0
Total params        : 5929556
Trainable params    : 7684
Non-trainable params: 5921904


Model: "efficientnetv2b0_4class"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape       ┃     Param # ┃ Connected to       ┃ Train… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ image_input           │ (None, 224, 224,   │           0 │ -                  │   -    │
│ (InputLayer)          │ 3)                 │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ rescaling (Rescaling) │ (None, 224, 224,   │           0 │ image_input[0][0]  │   -    │
│                       │ 3)                 │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ normalization         │ (None, 224, 224,   │           0 │ rescaling[0][0]    │   -    │
│ (Normalization)       │ 3)                 │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ stem_conv (Conv2D)    │ (None, 112, 112,   │         864 │ normalization[0][… │   N    │
│                       │ 32)                │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ stem_bn               │ (None, 112, 112,   │         128 │ stem_conv[0][0]    │   N    │
│ (BatchNormalization)  │ 32)                │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ stem_activation       │ (None, 112, 112,   │           0 │ stem_bn[0][0]      │   -    │
│ (Activation)          │ 32)                │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ block1a_project_conv  │ (None, 112, 112,   │       4,608 │ stem_activation[0… │   N    │
│ (Conv2D)              │ 16)                │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ block1a_project_bn    │ (None, 112, 112,   │          64 │ block1a_project_c… │   N    │
│ (BatchNormalization)  │ 16)                │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ block1a_project_acti… │ (None, 112, 112,   │           0 │ block1a_project_b… │   -    │
│ (Activation)          │ 16)                │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ block2a_expand_conv   │ (None, 56, 56, 64) │       9,216 │ block1a_project_a… │   N    │
│ (Conv2D)              │                    │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ block2a_expand_bn     │ (None, 56, 56, 64) │         256 │ block2a_expand_co… │   N    │
│ (BatchNormalization)  │                    │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ block2a_expand_activ… │ (None, 56, 56, 64) │           0 │ block2a_expand_bn… │   -    │
│ (Activation)          │                    │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ block2a_project_conv  │ (None, 56, 56, 32) │       2,048 │ block2a_expand_ac… │   N    │
│ (Conv2D)              │                    │             │                    │        │
├───────────────────────┼────────────────────┼─────────────┼────────────────────┼────────┤
│ block2a_project_bn    │ (None, 56, 56, 32) │         128 │ block2a_project_c… │   N    │
│ (BatchNormalization)  │                    │             │                    │      

 Total params: 5,929,556 (22.62 MB)

 Trainable params: 7,684 (30.02 KB)

 Non-trainable params: 5,921,872 (22.59 MB)

## Stage 4 — Phase 1: Train classification head (backbone frozen)

In [6]:
# Cell 5 — Phase 1: freeze backbone, train head
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=PHASE1_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_p1 = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor='val_accuracy', mode='max',
        save_best_only=True, verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=6,
        restore_best_weights=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1,
    ),
    keras.callbacks.CSVLogger(str(LOG_DIR / 'phase1_log.csv'), append=False),
]

print(f'Phase 1: training head for up to {PHASE1_EPOCHS} epochs ...')
history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_p1,
    verbose=1,
)

print('Phase 1 complete.')
print(f'Best val_accuracy so far: {max(history_p1.history["val_accuracy"]):.4f}')


Phase 1: training head for up to 20 epochs ...
Epoch 1/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 901ms/step - accuracy: 0.5919 - loss: 0.9640
Epoch 1: val_accuracy improved from None to 0.67837, saving model to d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras

Epoch 1: finished saving model to d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras
129/129 ━━━━━━━━━━━━━━━━━━━━ 153s 1s/step - accuracy: 0.5919 - loss: 0.9640 - val_accuracy: 0.6784 - val_loss: 0.8080 - learning_rate: 0.0010
Epoch 2/20
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 737ms/step - accuracy: 0.6956 - loss: 0.6555
Epoch 2: val_accuracy improved from 0.67837 to 0.76557, saving model to d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras

Epoch 2: finished saving model to d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras
129/129 ━━━━━━━━━━━━━━━━━━━━ 114s 881ms/step - accuracy: 0.6956 - loss: 0.6555 - val_accuracy: 0.765

## Stage 5 — Phase 2: Fine-tune upper backbone layers

In [7]:
# Cell 6 — Phase 2: unfreeze upper backbone, fine-tune
# Unfreeze the backbone but keep BatchNormalization layers frozen
# to preserve stable batch statistics from ImageNet pretraining.
backbone.trainable = True
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Only fine-tune the top portion (last ~40% of backbone layers)
total_layers = len(backbone.layers)
freeze_until = int(total_layers * 0.60)
for layer in backbone.layers[:freeze_until]:
    layer.trainable = False

trainable_p2 = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f'Backbone total layers : {total_layers}')
print(f'Frozen up to layer    : {freeze_until}')
print(f'Trainable params P2   : {trainable_p2:,}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=PHASE2_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_p2 = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor='val_accuracy', mode='max',
        save_best_only=True, verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8,
        restore_best_weights=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4,
        min_lr=1e-7, verbose=1,
    ),
    keras.callbacks.CSVLogger(str(LOG_DIR / 'phase2_log.csv'), append=False),
]

print(f'Phase 2: fine-tuning for up to {PHASE2_EPOCHS} epochs ...')
history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_p2,
    verbose=1,
)

print('Phase 2 complete.')
print(f'Best val_accuracy overall: {max(history_p2.history["val_accuracy"]):.4f}')

# Save final model
model.save(str(FINAL_MODEL_PATH))
print('Final model saved to:', FINAL_MODEL_PATH)


Backbone total layers : 270
Frozen up to layer    : 162
Trainable params P2   : 4,205,140
Phase 2: fine-tuning for up to 40 epochs ...
Epoch 1/40
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7759 - loss: 0.4656
Epoch 1: val_accuracy improved from None to 0.81993, saving model to d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras

Epoch 1: finished saving model to d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras
129/129 ━━━━━━━━━━━━━━━━━━━━ 210s 1s/step - accuracy: 0.7759 - loss: 0.4656 - val_accuracy: 0.8199 - val_loss: 0.5066 - learning_rate: 1.0000e-05
Epoch 2/40
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7827 - loss: 0.4502
Epoch 2: val_accuracy improved from 0.81993 to 0.82220, saving model to d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras

Epoch 2: finished saving model to d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras
129/129 ━━━━━━

## Stage 6 — Training history and learning curves

In [8]:
# Cell 7 — Merge Phase 1 + Phase 2 history and plot
def merge_history(h1, h2):
    merged = {}
    for k in h1.history:
        merged[k] = h1.history[k] + h2.history[k]
    return merged

full_history = merge_history(history_p1, history_p2)
hist_df = pd.DataFrame(full_history)
hist_df.index.name = 'epoch'
hist_df.to_csv(HISTORY_CSV)
print(f'Training history saved to {HISTORY_CSV}')
print(hist_df.tail(5).to_string())

# Plot accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
p1_end = len(history_p1.history['accuracy'])

for ax, metric, title in zip(axes,
                              [('accuracy','val_accuracy'), ('loss','val_loss')],
                              ['Accuracy', 'Loss']):
    train_key, val_key = metric
    ax.plot(full_history[train_key], label='Train')
    ax.plot(full_history[val_key],   label='Validation')
    ax.axvline(p1_end - 1, color='gray', linestyle='--', alpha=0.7, label='Phase 1→2')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.set_title(f'Training {title}')
    ax.legend()

plt.tight_layout()
plt.savefig(REPORT_DIR / 'training_curves_v2.png', dpi=120)
plt.show()
print('Saved training_curves_v2.png')


Training history saved to d:\Practice Projects\Disease Detection\model\training_history_v2.csv
       accuracy      loss  val_accuracy  val_loss  learning_rate
epoch                                                           
34     0.829813  0.323669      0.844847  0.396387       0.000005
35     0.832484  0.324982      0.847112  0.405353       0.000005
36     0.832484  0.329194      0.842582  0.396902       0.000005
37     0.837339  0.306376      0.843715  0.395838       0.000005
38     0.837582  0.319322      0.844847  0.390371       0.000005
Saved training_curves_v2.png


## Stage 7 — Evaluate best model on held-out test set

In [9]:
# Cell 8 — Load best saved model and run test evaluation
print('Loading best model from:', BEST_MODEL_PATH)
best_model = keras.models.load_model(str(BEST_MODEL_PATH))

test_loss, test_acc = best_model.evaluate(test_ds, verbose=1)
print(f'\nTest loss    : {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')


Loading best model from: d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras
28/28 ━━━━━━━━━━━━━━━━━━━━ 24s 626ms/step - accuracy: 0.8381 - loss: 0.4263

Test loss    : 0.4263
Test accuracy: 0.8381 (83.81%)


## Stage 8 — Classification report and confusion matrix

In [10]:
# Cell 9 — Predictions, classification report, confusion matrix
y_true, y_pred_probs = [], []
for imgs, lbls in test_ds:
    probs = best_model.predict(imgs, verbose=0)
    y_pred_probs.append(probs)
    y_true.extend(lbls.numpy())

y_pred_probs = np.concatenate(y_pred_probs, axis=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.array(y_true)

class_labels = [CLASS_NAMES[i] for i in sorted(CLASS_NAMES)]

# Classification report
report_str  = classification_report(y_true, y_pred, target_names=class_labels)
report_dict = classification_report(y_true, y_pred, target_names=class_labels, output_dict=True)
print('Classification Report:')
print(report_str)

report_df = pd.DataFrame(report_dict).T
report_df.to_csv(REPORT_DIR / 'classification_report.csv')
print('Saved classification_report.csv')

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=class_labels, columns=class_labels)
cm_df.to_csv(REPORT_DIR / 'confusion_matrix.csv')
print('Saved confusion_matrix.csv')

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'confusion_matrix.png', dpi=120)
plt.show()
print('Saved confusion_matrix.png')


Classification Report:
                      precision    recall  f1-score   support

              Normal       0.77      0.92      0.84       342
            Cataract       0.88      0.92      0.90       156
Diabetic Retinopathy       0.95      0.95      0.95       165
            Glaucoma       0.87      0.56      0.68       220

            accuracy                           0.84       883
           macro avg       0.87      0.84      0.84       883
        weighted avg       0.85      0.84      0.83       883

Saved classification_report.csv
Saved confusion_matrix.csv
Saved confusion_matrix.png


## Stage 9 — Per-class metrics and metrics JSON

In [11]:
# Cell 10 — Per-class accuracy/recall and save metrics JSON
per_class_rows = []
for cid in sorted(CLASS_NAMES):
    mask = y_true == cid
    n_total   = int(mask.sum())
    n_correct = int((y_pred[mask] == cid).sum())
    acc = n_correct / n_total if n_total > 0 else 0.0
    per_class_rows.append({
        'class_id': cid,
        'disease_name': CLASS_NAMES[cid],
        'n_samples': n_total,
        'n_correct': n_correct,
        'accuracy': round(acc, 4),
        'precision': round(report_dict[CLASS_NAMES[cid]]['precision'], 4),
        'recall':    round(report_dict[CLASS_NAMES[cid]]['recall'],    4),
        'f1_score':  round(report_dict[CLASS_NAMES[cid]]['f1-score'],  4),
    })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(REPORT_DIR / 'per_class_metrics.csv', index=False)
print(per_class_df.to_string(index=False))

# Save metrics JSON
metrics = {
    'test_loss': round(float(test_loss), 4),
    'test_accuracy': round(float(test_acc), 4),
    'per_class': per_class_rows,
    'macro_avg': {k: round(report_dict['macro avg'][k], 4)
                  for k in ['precision','recall','f1-score']},
    'weighted_avg': {k: round(report_dict['weighted avg'][k], 4)
                     for k in ['precision','recall','f1-score']},
}
(REPORT_DIR / 'test_metrics.json').write_text(
    json.dumps(metrics, indent=2), encoding='utf-8')
print('\nSaved test_metrics.json')


 class_id         disease_name  n_samples  n_correct  accuracy  precision  recall  f1_score
        0               Normal        342        315    0.9211     0.7664  0.9211    0.8367
        1             Cataract        156        144    0.9231     0.8780  0.9231    0.9000
        2 Diabetic Retinopathy        165        157    0.9515     0.9515  0.9515    0.9515
        3             Glaucoma        220        124    0.5636     0.8671  0.5636    0.6832

Saved test_metrics.json


## Stage 10 — predict_disease helper and sample verification

In [12]:
# Cell 11 — predict_disease helper function
def predict_disease(image_path, model=best_model):
    """Return disease_name, class_id, confidence, and all class probabilities."""
    raw  = tf.io.read_file(str(image_path))
    img  = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img  = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img  = tf.cast(img, tf.float32)
    img  = tf.expand_dims(img, 0)          # add batch dim
    probs = model.predict(img, verbose=0)[0]
    class_id = int(np.argmax(probs))
    return {
        'disease_name': CLASS_NAMES[class_id],
        'class_id': class_id,
        'confidence': round(float(probs[class_id]), 4),
        'all_probabilities': {
            CLASS_NAMES[i]: round(float(probs[i]), 4)
            for i in sorted(CLASS_NAMES)
        },
    }

# Sample 8 test images (2 per class) and verify output
print('Sample predictions from test set:')
print('-' * 70)
shown = {cid: 0 for cid in CLASS_NAMES}
for _, row in test_df.sample(frac=1, random_state=SEED).iterrows():
    cid = row['class_id']
    if shown[cid] >= 2:
        continue
    result = predict_disease(row['image_path'])
    true_name = CLASS_NAMES[cid]
    status = 'OK' if result['disease_name'] == true_name else 'WRONG'
    print(f"[{status}] True: {true_name:<22} "
          f"Pred: {result['disease_name']:<22} "
          f"Conf: {result['confidence']:.3f}")
    shown[cid] += 1
    if all(v >= 2 for v in shown.values()):
        break


Sample predictions from test set:
----------------------------------------------------------------------
[OK] True: Cataract               Pred: Cataract               Conf: 1.000
[OK] True: Diabetic Retinopathy   Pred: Diabetic Retinopathy   Conf: 0.999
[OK] True: Diabetic Retinopathy   Pred: Diabetic Retinopathy   Conf: 1.000
[OK] True: Cataract               Pred: Cataract               Conf: 1.000
[OK] True: Normal                 Pred: Normal                 Conf: 0.948
[OK] True: Normal                 Pred: Normal                 Conf: 0.962
[WRONG] True: Glaucoma               Pred: Diabetic Retinopathy   Conf: 0.871
[OK] True: Glaucoma               Pred: Glaucoma               Conf: 0.995


## Stage 11 — Save model metadata

In [13]:
# Cell 12 — Save model_metadata_v2.json
metadata = {
    'model_name': 'efficientnetv2b0_4class',
    'architecture': 'EfficientNetV2-B0',
    'input_size': [IMG_SIZE, IMG_SIZE, 3],
    'num_classes': NUM_CLASSES,
    'class_id_to_name': {str(k): v for k, v in CLASS_NAMES.items()},
    'preprocessing': {
        'include_preprocessing': True,
        'pixel_range': '[0, 255] float32',
        'note': 'EfficientNetV2B0 include_preprocessing=True handles internal scaling',
    },
    'training_dataset': {
        'train_csv': str(TRAIN_CSV),
        'val_csv':   str(VAL_CSV),
        'test_csv':  str(TEST_CSV),
    },
    'training_config': {
        'seed': SEED,
        'batch_size': BATCH_SIZE,
        'phase1_lr': PHASE1_LR,
        'phase2_lr': PHASE2_LR,
        'phase1_epochs_max': PHASE1_EPOCHS,
        'phase2_epochs_max': PHASE2_EPOCHS,
        'dropout_rate': DROPOUT_RATE,
        'class_weights': CLASS_WEIGHTS,
        'augmentation': ['RandomFlip(horizontal)', 'RandomRotation(0.05)',
                         'RandomZoom(0.10)', 'RandomContrast(0.10)'],
    },
    'saved_models': {
        'best_model':  str(BEST_MODEL_PATH),
        'final_model': str(FINAL_MODEL_PATH),
    },
    'backbone_layer_name': backbone_name,
    'gradcam_target_layer': 'top_activation',
    'test_metrics': metrics,
}

METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Model metadata saved to:', METADATA_PATH)
print(json.dumps({k: metadata[k] for k in
                  ['model_name','num_classes','class_id_to_name',
                   'backbone_layer_name','gradcam_target_layer']}, indent=2))


Model metadata saved to: d:\Practice Projects\Disease Detection\model\model_metadata_v2.json
{
  "model_name": "efficientnetv2b0_4class",
  "num_classes": 4,
  "class_id_to_name": {
    "0": "Normal",
    "1": "Cataract",
    "2": "Diabetic Retinopathy",
    "3": "Glaucoma"
  },
  "backbone_layer_name": "efficientnetv2-b0",
  "gradcam_target_layer": "top_activation"
}


## MODEL TRAINING V2 — COMPLETED

In [14]:
# Cell 13 — Final summary
print('=' * 65)
print('  MODEL TRAINING V2 COMPLETED')
print('=' * 65)
print(f'  Best model path  : {BEST_MODEL_PATH}')
print(f'  Final model path : {FINAL_MODEL_PATH}')
print(f'  Test loss        : {test_loss:.4f}')
print(f'  Test accuracy    : {test_acc*100:.2f}%')
print()
print('  Class mapping:')
for cid, name in CLASS_NAMES.items():
    print(f'    {cid} = {name}')
print()
print('  Per-class test metrics:')
print(per_class_df[['class_id','disease_name','n_samples',
                     'accuracy','precision','recall','f1_score']].to_string(index=False))
print()
print('  Generated reports:')
for f in sorted(REPORT_DIR.iterdir()):
    print(f'    {f}')
print()
print('  Training history :', HISTORY_CSV)
print('  Model metadata   :', METADATA_PATH)
print('=' * 65)


  MODEL TRAINING V2 COMPLETED
  Best model path  : d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_best.keras
  Final model path : d:\Practice Projects\Disease Detection\model\efficientnetv2_b0_4class_final.keras
  Test loss        : 0.4263
  Test accuracy    : 83.81%

  Class mapping:
    0 = Normal
    1 = Cataract
    2 = Diabetic Retinopathy
    3 = Glaucoma

  Per-class test metrics:
 class_id         disease_name  n_samples  accuracy  precision  recall  f1_score
        0               Normal        342    0.9211     0.7664  0.9211    0.8367
        1             Cataract        156    0.9231     0.8780  0.9231    0.9000
        2 Diabetic Retinopathy        165    0.9515     0.9515  0.9515    0.9515
        3             Glaucoma        220    0.5636     0.8671  0.5636    0.6832

  Generated reports:
    d:\Practice Projects\Disease Detection\reports\model_v2\classification_report.csv
    d:\Practice Projects\Disease Detection\reports\model_v2\confusion_mat